# Small-Model Alignment - Kaggle Orchestrator

This notebook intentionally contains **no training logic**. It configures a Kaggle T4 runtime and calls the version-controlled scripts in this repository. Run one expensive stage at a time, save a Kaggle version afterward, then attach its outputs to the next notebook/stage.

Before running: enable Internet for Hugging Face downloads (or attach cached models/datasets), select **GPU T4 x2**, and set `REPO_URL` below if this notebook is not already inside the cloned repository.

In [ ]:
# ---- User configuration ----
REPO_URL = "https://github.com/YOUR_USER/model-alignment.git"  
REPO_DIR = "/kaggle/working/model-alignment"

# Attach your prepared JSON/JSONL datasets as Kaggle inputs, then set these paths.
SFT_TRAIN = "/kaggle/input/alignment-data/sft_train.jsonl"
SFT_VALID = "/kaggle/input/alignment-data/sft_valid.jsonl"
HH_TRAIN = "/kaggle/input/alignment-data/hh_train.jsonl"
HH_VALID = "/kaggle/input/alignment-data/hh_valid.jsonl"
UF_TRAIN = "/kaggle/input/alignment-data/ultrafeedback_train.jsonl"
UF_VALID = "/kaggle/input/alignment-data/ultrafeedback_valid.jsonl"
ALPACA_EVAL = "/kaggle/input/alignment-data/alpaca_eval_300_seed42.json"

# Dataset field mapping. Change only if your data preparation uses other names.
SFT_PROMPT_FIELD, SFT_RESPONSE_FIELD = "prompt", "response"
DPO_PROMPT_FIELD, DPO_CHOSEN_FIELD, DPO_REJECTED_FIELD = "prompt", "chosen", "rejected"
# Keep this exact template stable across SFT, DPO, and generation.
PROMPT_TEMPLATE = "{prompt}"

# Select stages deliberately. Save a Kaggle version after each expensive stage.
RUN_SFT = False
RUN_HH_DPO = False
RUN_ULTRAFEEDBACK_DPO = False
RUN_PERSONA_LORA = False
RUN_GENERATION = False
RUN_PAIR_BUILD = False


In [ ]:
import os
import subprocess
from pathlib import Path

if REPO_URL:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Using existing repository: {REPO_DIR}")
else:
    # For an uploaded notebook, set this to the directory containing submission_adapter.py.
    REPO_DIR = os.getcwd() if Path("submission_adapter.py").exists() else REPO_DIR

repo = Path(REPO_DIR).resolve()
assert (repo / "submission_adapter.py").exists(), f"Repository not found at {repo}; set REPO_URL or REPO_DIR."
os.chdir(repo)
print(f"Working from {repo}")

# Preserve Kaggle's CUDA-compatible PyTorch. Do not install/reinstall torch.
subprocess.run(["python", "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
import datasets
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("GPU count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
assert torch.cuda.device_count() >= 2, "Select GPU T4 x2 for the two-GPU DPO recipe."
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["python", "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], check=True, env={**os.environ, "PYTHONPATH": "."})


In [ ]:
# Shared paths. Each stage writes metrics continuously plus best/final checkpoints.
OUT = Path("/kaggle/working/model-alignment-outputs")
SFT_OUT = OUT / "sft"
HH_OUT = OUT / "hh_dpo"
UF_OUT = OUT / "ultrafeedback_dpo"
PERSONA_OUT = OUT / "persona_lora"
GEN_OUT = OUT / "generations"
EVAL_OUT = OUT / "eval"
OUT.mkdir(parents=True, exist_ok=True)

def run(command):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(value) for value in command], check=True, cwd=repo)


In [ ]:
if RUN_SFT:
    run(["python", "scripts/train_sft.py",
         "--train", SFT_TRAIN, "--validation", SFT_VALID, "--output_dir", SFT_OUT,
         "--prompt_field", SFT_PROMPT_FIELD, "--response_field", SFT_RESPONSE_FIELD,
         "--prompt_template", PROMPT_TEMPLATE, "--max_length", 512,
         "--batch_size", 2, "--gradient_accumulation", 16, "--learning_rate", 2e-5])


In [ ]:
def run_dpo(train_path, valid_path, output_dir):
    run(["python", "scripts/train_dpo.py",
         "--sft_checkpoint", SFT_OUT / "best", "--train", train_path, "--validation", valid_path,
         "--output_dir", output_dir, "--prompt_field", DPO_PROMPT_FIELD,
         "--chosen_field", DPO_CHOSEN_FIELD, "--rejected_field", DPO_REJECTED_FIELD,
         "--prompt_template", PROMPT_TEMPLATE, "--max_length", 512, "--batch_size", 1,
         "--gradient_accumulation", 32, "--learning_rate", 1e-6, "--beta", 0.1])

if RUN_HH_DPO:
    run_dpo(HH_TRAIN, HH_VALID, HH_OUT)
if RUN_ULTRAFEEDBACK_DPO:
    run_dpo(UF_TRAIN, UF_VALID, UF_OUT)
if RUN_PERSONA_LORA:
    run(["python", "scripts/train_persona_lora.py", "--output_dir", PERSONA_OUT])


In [ ]:
# Produce comparable SFT/DPO outputs with identical greedy decoding settings.
# For HH, replace ALPACA_EVAL with `eval_sets/hh_rlhf_eval_300_seed42.json`.
if RUN_GENERATION:
    for checkpoint, name in [(SFT_OUT / "best", "sft"), (UF_OUT / "best", "ultrafeedback_dpo")]:
        run(["python", "scripts/generate.py", "--model", checkpoint, "--inputs", ALPACA_EVAL,
             "--output", GEN_OUT / f"{name}_alpaca_eval.json", "--generator", name,
             "--dataset", "alpaca_eval_seed42", "--prompt_template", PROMPT_TEMPLATE,
             "--max_new_tokens", 256])

if RUN_PAIR_BUILD:
    run(["python", "scripts/build_pairwise.py",
         "--sft", GEN_OUT / "sft_alpaca_eval.json",
         "--dpo", GEN_OUT / "ultrafeedback_dpo_alpaca_eval.json",
         "--output", EVAL_OUT / "ultrafeedback_dpo_vs_sft_pairs.jsonl", "--seed", 42])

print("Outputs ready at:", OUT)
print("Save a Kaggle version now. Run model judging in a separate Kaggle benchmark notebook from the saved pairwise JSONL.")
